# 🧠 Redes Neuronales con Python — 5 Casos de la Vida Real

> **Curso:** Sistemas Inteligentes y Machine Learning  
> **Universidad Privada del Norte (UPN)**  
> **Docente:** Steven  

---

Este notebook presenta **5 ejemplos prácticos y reales** de redes neuronales implementadas en Python usando **TensorFlow / Keras**.  
Cada ejemplo incluye: contexto del problema, preparación de datos, diseño de la red, entrenamiento y evaluación.

| # | Caso de Uso | Tipo de Red |
|---|-------------|-------------|
| 1 | Diagnóstico de diabetes (Pima Indians) | Clasificación binaria |
| 2 | Predicción del precio de casas | Regresión |
| 3 | Clasificación de imágenes de ropa | CNN |
| 4 | Detección de spam en correos | NLP + Embedding |
| 5 | Predicción de ventas futuras | LSTM (Series de tiempo) |

---


## ⚙️ Instalación de dependencias

In [ ]:
# Ejecuta esta celda solo si no tienes las librerías instaladas
# !pip install tensorflow numpy pandas matplotlib scikit-learn seaborn

import warnings
warnings.filterwarnings('ignore')
print("✅ Librerías listas")


---
## 🩺 Ejemplo 1: Diagnóstico de Diabetes

### Contexto Real
El Hospital Nacional utilizó datos clínicos de pacientes para predecir si una persona tiene diabetes.  
Se usa el famoso dataset **Pima Indians Diabetes** (768 pacientes, 8 variables clínicas).

### Variables de entrada
- Glucosa en sangre, presión arterial, índice de masa corporal (IMC)
- Edad, número de embarazos, insulina, grosor de piel, función del pedigrí

### Objetivo
Predecir si el paciente tiene diabetes (`1`) o no (`0`).

### Arquitectura de la Red
```
Entrada (8) → Capa Densa (16, ReLU) → Capa Densa (8, ReLU) → Salida (1, Sigmoid)
```


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

# ----------------------------------------------------------
# PASO 1: Cargar datos
# ----------------------------------------------------------
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Embarazos", "Glucosa", "PresionArterial", "GrosorPiel",
        "Insulina", "IMC", "Pedigri", "Edad", "Diabetes"]

try:
    df = pd.read_csv(url, names=cols)
except:
    # Dataset sintético si no hay internet
    np.random.seed(0)
    n = 768
    df = pd.DataFrame({
        "Embarazos": np.random.randint(0, 10, n),
        "Glucosa": np.random.normal(120, 30, n).clip(50, 200),
        "PresionArterial": np.random.normal(70, 12, n).clip(40, 120),
        "GrosorPiel": np.random.normal(20, 10, n).clip(0, 60),
        "Insulina": np.random.normal(80, 60, n).clip(0, 400),
        "IMC": np.random.normal(32, 7, n).clip(15, 60),
        "Pedigri": np.random.uniform(0.1, 2.0, n),
        "Edad": np.random.randint(21, 70, n),
        "Diabetes": np.random.randint(0, 2, n)
    })

print(f"📊 Dataset cargado: {df.shape[0]} pacientes, {df.shape[1]-1} variables")
print(f"   Positivos (diabetes): {df['Diabetes'].sum()} | Negativos: {(df['Diabetes']==0).sum()}")
df.head()


In [ ]:
# ----------------------------------------------------------
# PASO 2: Preprocesamiento
# ----------------------------------------------------------
X = df.drop("Diabetes", axis=1).values
y = df["Diabetes"].values

# Normalizar features (importante para redes neuronales)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División entrenamiento / prueba (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"🔀 Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")

# ----------------------------------------------------------
# PASO 3: Construir la Red Neuronal
# ----------------------------------------------------------
model1 = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(8,), name="capa_oculta_1"),
    tf.keras.layers.Dropout(0.3),                                     # Regularización
    tf.keras.layers.Dense(8,  activation='relu', name="capa_oculta_2"),
    tf.keras.layers.Dense(1,  activation='sigmoid', name="salida")   # Probabilidad 0-1
], name="Red_Diabetes")

model1.summary()


In [ ]:
# ----------------------------------------------------------
# PASO 4: Compilar y Entrenar
# ----------------------------------------------------------
model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',   # Pérdida para clasificación binaria
    metrics=['accuracy']
)

history1 = model1.fit(
    X_train, y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.15,
    verbose=0
)

# ----------------------------------------------------------
# PASO 5: Evaluación y Visualización
# ----------------------------------------------------------
loss, acc = model1.evaluate(X_test, y_test, verbose=0)
print(f"\n📈 Precisión en datos de prueba: {acc:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Curvas de aprendizaje
axes[0].plot(history1.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[0].plot(history1.history['val_accuracy'], label='Val Accuracy',   color='orange', linestyle='--')
axes[0].set_title('Curvas de Precisión — Diagnóstico de Diabetes', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Precisión')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Matriz de confusión
y_pred = (model1.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No Diabetes','Diabetes'],
            yticklabels=['No Diabetes','Diabetes'])
axes[1].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicción'); axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

print("\n📋 Reporte detallado:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))


---
## 🏠 Ejemplo 2: Predicción del Precio de Casas

### Contexto Real
Una inmobiliaria en Lima necesita estimar el precio de venta de propiedades automáticamente.  
Se usa el dataset **California Housing** (20,640 bloques censales con características de vivienda).

### Variables de entrada
Ingreso medio de la zona, antigüedad de las casas, habitaciones, dormitorios, población, hogares, latitud/longitud.

### Objetivo
Predecir el **valor mediano de la propiedad** (en miles de dólares).

### Arquitectura
```
Entrada (8) → Dense(64, ReLU) → Dense(32, ReLU) → Dense(16, ReLU) → Salida (1, lineal)
```


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, r2_score

# ----------------------------------------------------------
# PASO 1: Cargar datos
# ----------------------------------------------------------
housing = fetch_california_housing()
X2 = housing.data
y2 = housing.target   # Precio en cientos de miles de dólares

print(f"📊 Dataset: {X2.shape[0]:,} registros, {X2.shape[1]} características")
print(f"   Precio mínimo: ${y2.min()*100:.0f}k | Máximo: ${y2.max()*100:.0f}k | Promedio: ${y2.mean()*100:.0f}k")

# ----------------------------------------------------------
# PASO 2: Preprocesar
# ----------------------------------------------------------
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2_scaled, y2, test_size=0.2, random_state=42)

# ----------------------------------------------------------
# PASO 3: Construir la Red Neuronal de Regresión
# ----------------------------------------------------------
model2 = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(8,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)    # Salida lineal para regresión
], name="Red_Precios_Casas")

model2.compile(optimizer=tf.keras.optimizers.Adam(0.001),
               loss='mse',          # Error cuadrático medio
               metrics=['mae'])     # Error absoluto medio

# ----------------------------------------------------------
# PASO 4: Entrenar con Early Stopping
# ----------------------------------------------------------
early_stop = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

history2 = model2.fit(
    X2_train, y2_train,
    epochs=100, batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0
)
print(f"✅ Entrenamiento completado en {len(history2.history['loss'])} épocas")


In [ ]:
# ----------------------------------------------------------
# PASO 5: Evaluación
# ----------------------------------------------------------
y2_pred = model2.predict(X2_test, verbose=0).flatten()
mae  = mean_absolute_error(y2_test, y2_pred)
r2   = r2_score(y2_test, y2_pred)

print(f"📉 Error Absoluto Medio (MAE): ${mae*100:.0f}k")
print(f"📊 R² Score: {r2:.4f}  (1.0 = perfecto)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Predicho vs Real
axes[0].scatter(y2_test, y2_pred, alpha=0.3, s=10, color='steelblue')
lims = [min(y2_test.min(), y2_pred.min()), max(y2_test.max(), y2_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Precio Real ($100k)'); axes[0].set_ylabel('Precio Predicho ($100k)')
axes[0].set_title('Real vs Predicho — Precio de Casas', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Curva de pérdida
axes[1].plot(history2.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history2.history['val_loss'], label='Val Loss',   color='orange', linestyle='--')
axes[1].set_title('Curva de Pérdida (MSE)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('MSE')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


---
## 👕 Ejemplo 3: Clasificación de Imágenes de Ropa (CNN)

### Contexto Real
Una tienda de e-commerce necesita clasificar automáticamente fotografías de productos textiles  
para organizar su catálogo. Se usa **Fashion MNIST** (70,000 imágenes 28×28 en escala de grises).

### Clases
T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle Boot.

### Arquitectura CNN
```
Imagen (28×28×1) → Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Flatten → Dense(128) → Salida(10)
```


In [ ]:
# ----------------------------------------------------------
# PASO 1: Cargar Fashion MNIST
# ----------------------------------------------------------
(X3_train, y3_train), (X3_test, y3_test) = tf.keras.datasets.fashion_mnist.load_data()

clases = ['Camiseta','Pantalón','Pullover','Vestido','Abrigo',
          'Sandalia','Camisa','Zapatilla','Bolso','Botín']

print(f"📦 Train: {X3_train.shape} | Test: {X3_test.shape}")

# Visualizar ejemplos
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X3_train[i], cmap='gray')
    ax.set_title(clases[y3_train[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion MNIST — Ejemplos del Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ----------------------------------------------------------
# PASO 2: Preprocesar para CNN
# ----------------------------------------------------------
X3_train_n = X3_train.reshape(-1, 28, 28, 1) / 255.0
X3_test_n  = X3_test.reshape(-1, 28, 28, 1)  / 255.0

# ----------------------------------------------------------
# PASO 3: Construir CNN
# ----------------------------------------------------------
model3 = tf.keras.Sequential([
    # Bloque convolucional 1
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D(2,2),

    # Bloque convolucional 2
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    # Clasificador
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax')   # 10 categorías
], name="CNN_Ropa")

model3.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])
model3.summary()


In [ ]:
# ----------------------------------------------------------
# PASO 4: Entrenar
# ----------------------------------------------------------
history3 = model3.fit(
    X3_train_n, y3_train,
    epochs=10, batch_size=128,
    validation_split=0.1,
    verbose=1
)

# ----------------------------------------------------------
# PASO 5: Evaluación y predicciones
# ----------------------------------------------------------
loss3, acc3 = model3.evaluate(X3_test_n, y3_test, verbose=0)
print(f"\n🎯 Precisión en test: {acc3:.2%}")

# Mostrar predicciones con errores marcados
y3_pred = np.argmax(model3.predict(X3_test_n, verbose=0), axis=1)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X3_test[i], cmap='gray')
    color = 'green' if y3_pred[i] == y3_test[i] else 'red'
    ax.set_title(f"R:{clases[y3_test[i]]}\nP:{clases[y3_pred[i]]}", fontsize=7, color=color)
    ax.axis('off')
plt.suptitle('Verde = Correcto | Rojo = Error', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 📧 Ejemplo 4: Detección de Spam en Correos Electrónicos

### Contexto Real
El área de TI de una empresa necesita filtrar correos spam automáticamente.  
Se usa una red neuronal con **capa de Embedding** para procesar texto.

### Flujo de procesamiento
```
Texto → Tokenización → Embedding (100 dims) → GlobalAveragePooling → Dense(16) → Salida
```


In [ ]:
# ----------------------------------------------------------
# PASO 1: Dataset de correos (sintético realista)
# ----------------------------------------------------------
correos_spam = [
    "Ganaste 1 millón de dólares haz clic aquí ahora",
    "OFERTA EXCLUSIVA precio especial solo hoy compra ya",
    "Tu cuenta fue bloqueada verifica tus datos urgente",
    "Gana dinero fácil trabajando desde casa sin esfuerzo",
    "GRATIS iPhone 15 solo por completar esta encuesta",
    "Préstamo inmediato sin requisitos apruébate en minutos",
    "Invierte en criptomonedas y gana miles esta semana",
    "Alerta tu tarjeta fue comprometida llama ya",
    "Premio especial para ti reclama tu regalo gratis",
    "Descuento 90% solo las próximas 2 horas no te pierdas",
    "Aumenta tus ingresos con este método secreto revelado",
    "Tu número fue seleccionado para recibir efectivo hoy",
]
correos_ham = [
    "Reunión de equipo el martes a las 10am sala de conferencias",
    "Adjunto el informe mensual de ventas para revisión",
    "Recuerda entregar el reporte antes del viernes por favor",
    "El cliente confirmó la reunión para el próximo lunes",
    "Actualización del sistema programada para este fin de semana",
    "Hola revisa los cambios al documento que compartí ayer",
    "Gracias por asistir a la capacitación de ayer estuvo muy bien",
    "Necesito tu opinión sobre la propuesta antes del jueves",
    "El proyecto avanza según lo planificado sin inconvenientes",
    "Confirmación de tu reserva de vuelo Lima Cusco el viernes",
    "La auditoría interna está programada para el próximo mes",
    "Por favor completa la encuesta de satisfacción del cliente",
]

textos = correos_spam + correos_ham
etiquetas = [1]*len(correos_spam) + [0]*len(correos_ham)  # 1=spam, 0=ham

print(f"📧 Dataset: {len(textos)} correos | {sum(etiquetas)} spam | {len(textos)-sum(etiquetas)} ham")

# ----------------------------------------------------------
# PASO 2: Tokenizar textos
# ----------------------------------------------------------
VOCAB_SIZE  = 1000
MAX_LEN     = 20

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(textos)
secuencias = tokenizer.texts_to_sequences(textos)
X4 = tf.keras.preprocessing.sequence.pad_sequences(secuencias, maxlen=MAX_LEN, padding='post')
y4 = np.array(etiquetas)

# División train/test
from sklearn.model_selection import train_test_split
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.3, random_state=42)
print(f"✅ Secuencias tokenizadas y divididas")


In [ ]:
# ----------------------------------------------------------
# PASO 3: Red Neuronal con Embedding
# ----------------------------------------------------------
model4 = tf.keras.Sequential([
    tf.keras.layers.Embedding(VOCAB_SIZE, 16, input_length=MAX_LEN),
    tf.keras.layers.GlobalAveragePooling1D(),    # Resume la secuencia en un vector
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
], name="Red_Spam_Detector")

model4.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model4.summary()

# ----------------------------------------------------------
# PASO 4: Entrenar
# ----------------------------------------------------------
history4 = model4.fit(
    X4_train, y4_train,
    epochs=40, batch_size=4,
    validation_data=(X4_test, y4_test),
    verbose=0
)

# ----------------------------------------------------------
# PASO 5: Evaluar y probar con ejemplos nuevos
# ----------------------------------------------------------
loss4, acc4 = model4.evaluate(X4_test, y4_test, verbose=0)
print(f"\n🎯 Precisión: {acc4:.2%}")

# Probar con correos nuevos
nuevos = [
    "Gana premios increíbles haz clic aquí gratis urgente",
    "Hola te confirmo nuestra reunión del jueves a las 3pm",
    "OFERTA última hora reclama tu dinero ahora mismo",
    "El informe de análisis está listo para tu revisión"
]

print("\n📬 Clasificación de correos nuevos:")
print(f"{'Correo':<52} {'Resultado':<12} {'Confianza'}")
print("-"*75)
for c in nuevos:
    seq = tokenizer.texts_to_sequences([c])
    pad = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    prob = model4.predict(pad, verbose=0)[0][0]
    lbl  = "🔴 SPAM" if prob > 0.5 else "✅ HAM"
    print(f"{c[:50]:<52} {lbl:<12} {prob:.1%}")


---
## 📈 Ejemplo 5: Predicción de Ventas con LSTM (Series de Tiempo)

### Contexto Real
Una cadena de farmacias en Lima quiere predecir sus ventas diarias para optimizar el inventario.  
Se usa una red **LSTM** (Long Short-Term Memory), especializada en secuencias temporales.

### Flujo
```
Histórico de ventas → Ventanas de 30 días → LSTM(50) → Dense(25) → Predicción del día siguiente
```


In [ ]:
# ----------------------------------------------------------
# PASO 1: Generar serie de tiempo de ventas (realista)
# ----------------------------------------------------------
np.random.seed(42)
dias = 500

t = np.arange(dias)

# Ventas con: tendencia + estacionalidad semanal + estacionalidad mensual + ruido
tendencia   = 1000 + 0.5 * t
semanal     = 150 * np.sin(2 * np.pi * t / 7)      # Picos los fines de semana
mensual     = 80  * np.sin(2 * np.pi * t / 30)      # Picos a fin de mes
ruido       = np.random.normal(0, 60, dias)
ventas      = tendencia + semanal + mensual + ruido

# Visualizar
plt.figure(figsize=(13, 4))
plt.plot(ventas[:120], color='steelblue', linewidth=1.2)
plt.title('Serie de Tiempo — Ventas Diarias de Farmacia (primeros 120 días)', fontsize=13, fontweight='bold')
plt.xlabel('Día'); plt.ylabel('Ventas (S/.)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"📊 Serie generada: {dias} días | Promedio: S/.{ventas.mean():.0f} | Std: S/.{ventas.std():.0f}")


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# ----------------------------------------------------------
# PASO 2: Preprocesar para LSTM
# ----------------------------------------------------------
VENTANA = 30    # Usamos los últimos 30 días para predecir el día 31

scaler5 = MinMaxScaler()
ventas_norm = scaler5.fit_transform(ventas.reshape(-1, 1))

def crear_ventanas(serie, ventana):
    X, y = [], []
    for i in range(len(serie) - ventana):
        X.append(serie[i:i+ventana])
        y.append(serie[i+ventana])
    return np.array(X), np.array(y)

X5, y5 = crear_ventanas(ventas_norm, VENTANA)
X5 = X5.reshape(X5.shape[0], X5.shape[1], 1)  # (muestras, pasos, features)

split = int(len(X5) * 0.8)
X5_train, X5_test = X5[:split], X5[split:]
y5_train, y5_test = y5[:split], y5[split:]

print(f"🔢 X_train: {X5_train.shape} | X_test: {X5_test.shape}")

# ----------------------------------------------------------
# PASO 3: Construir LSTM
# ----------------------------------------------------------
model5 = tf.keras.Sequential([
    tf.keras.layers.LSTM(50, return_sequences=True, input_shape=(VENTANA, 1)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(25),
    tf.keras.layers.Dense(1)
], name="LSTM_Prediccion_Ventas")

model5.compile(optimizer='adam', loss='mse')
model5.summary()


In [ ]:
# ----------------------------------------------------------
# PASO 4: Entrenar
# ----------------------------------------------------------
history5 = model5.fit(
    X5_train, y5_train,
    epochs=30, batch_size=16,
    validation_data=(X5_test, y5_test),
    verbose=0
)
print("✅ Entrenamiento completado")

# ----------------------------------------------------------
# PASO 5: Predicción y visualización
# ----------------------------------------------------------
y5_pred_norm = model5.predict(X5_test, verbose=0)

# Desnormalizar
y5_real_orig = scaler5.inverse_transform(y5_test)
y5_pred_orig = scaler5.inverse_transform(y5_pred_norm)

mae5 = mean_absolute_error(y5_real_orig, y5_pred_orig)
r2_5 = r2_score(y5_real_orig, y5_pred_orig)

print(f"\n📉 MAE: S/.{mae5:.2f} por día")
print(f"📊 R²:  {r2_5:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Real vs Predicho
dias_test = np.arange(len(y5_real_orig))
axes[0].plot(dias_test, y5_real_orig,  label='Ventas Reales',    color='steelblue', linewidth=1.2)
axes[0].plot(dias_test, y5_pred_orig,  label='Ventas Predichas', color='tomato',    linewidth=1.2, linestyle='--')
axes[0].set_title('LSTM — Predicción de Ventas Diarias', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Día de prueba'); axes[0].set_ylabel('Ventas (S/.)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Curva de pérdida
axes[1].plot(history5.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history5.history['val_loss'], label='Val Loss',   color='orange', linestyle='--')
axes[1].set_title('Curva de Pérdida LSTM', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('MSE')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Predicción del próximo día
ultimo_bloque = ventas_norm[-VENTANA:].reshape(1, VENTANA, 1)
prox_dia_norm = model5.predict(ultimo_bloque, verbose=0)
prox_dia = scaler5.inverse_transform(prox_dia_norm)[0][0]
print(f"\n🔮 Predicción ventas próximo día: S/.{prox_dia:.2f}")


---
## 📚 Resumen: ¿Qué aprendimos?

| # | Caso | Tipo de Red | Función de Pérdida | Activación Salida |
|---|------|-------------|-------------------|-------------------|
| 1 | Diagnóstico Diabetes | MLP | Binary Crossentropy | Sigmoid |
| 2 | Precio de Casas | MLP + BN | MSE | Lineal |
| 3 | Clasificación Ropa | CNN | Categorical Crossentropy | Softmax |
| 4 | Detección Spam | Embedding + Pool | Binary Crossentropy | Sigmoid |
| 5 | Predicción Ventas | LSTM | MSE | Lineal |

---

### 🔑 Conceptos Clave

- **Dense (MLP):** Capas completamente conectadas, base de casi toda red neuronal.
- **Conv2D (CNN):** Detecta patrones locales en imágenes (bordes, texturas, formas).
- **Embedding:** Convierte palabras en vectores numéricos de significado.
- **LSTM:** Recuerda dependencias largas en datos secuenciales (texto, tiempo).
- **Dropout:** Regularización que apaga neuronas al azar para evitar overfitting.
- **BatchNorm:** Normaliza activaciones para acelerar el entrenamiento.

### 🚀 Próximos pasos
- Probar con datasets propios (datos de tu hospital, empresa o ciudad)
- Ajustar hiperparámetros: épocas, neuronas, tasa de aprendizaje
- Explorar Transfer Learning con modelos preentrenados (VGG16, BERT)

---
*Notebook generado para el curso Sistemas Inteligentes y Machine Learning — UPN*
